In [1]:
# 1. Install & Import Libraries
# ==============================
!pip install torch torchvision --quiet

import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from google.colab import drive

# ==============================
# 2. Mount Google Drive
# ==============================
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# ==============================
# 3. Dataset Path
# ==============================
# Your professor/friend already has "Primary Categories" dataset in Drive
base_dir = "/content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories"

if not os.path.exists(base_dir):
    raise FileNotFoundError(f"❌ Dataset not found at {base_dir}")

print("✅ Found dataset at:", base_dir)

✅ Found dataset at: /content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories


In [2]:
# ==============================
# 3. Dataset Path
# ==============================
# Your professor/friend already has "Primary Categories" dataset in Drive
base_dir = "/content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories"

if not os.path.exists(base_dir):
    raise FileNotFoundError(f"❌ Dataset not found at {base_dir}")

print("✅ Found dataset at:", base_dir)
# ==============================
# 4. Prepare Dataset (Train/Val Split)
# ==============================
dataset_dir = "/content/dataset_cls"
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Loop through primary categories (Rice, BBQ, etc.)
for primary_cat in os.listdir(base_dir):
    primary_path = os.path.join(base_dir, primary_cat)
    if not os.path.isdir(primary_path):
        continue

    # Loop through food-item subfolders (Biryani, Chapli Kebab, etc.)
    for food_item in os.listdir(primary_path):
        food_item_path = os.path.join(primary_path, food_item)
        if not os.path.isdir(food_item_path):
            continue

        images = [f for f in os.listdir(food_item_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(images) == 0:
            continue

        # Split (80% train, 20% val)
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

        os.makedirs(os.path.join(train_dir, food_item), exist_ok=True)
        os.makedirs(os.path.join(val_dir, food_item), exist_ok=True)

        # Copy files
        for img in train_imgs:
            shutil.copy(os.path.join(food_item_path, img), os.path.join(train_dir, food_item, img))
        for img in val_imgs:
            shutil.copy(os.path.join(food_item_path, img), os.path.join(val_dir, food_item, img))

print("✅ Dataset prepared for ResNet50 classification.")




✅ Found dataset at: /content/drive/MyDrive/Smart_Food_Analyzer-Food_Images/Primary Categories


KeyboardInterrupt: 

In [3]:
# 5. Data Transforms
# ==============================
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
val_dataset   = datasets.ImageFolder(val_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print("🍴 Food classes:", class_names)

# ==============================

NameError: name 'transforms' is not defined

In [2]:
# 6. Load Pretrained ResNet50
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("⚡ Using device:", device)

model = models.resnet50(pretrained=True)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)


⚡ Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 120MB/s]


NameError: name 'class_names' is not defined

In [ ]:
# ==============================
# 7. Loss & Optimizer
# ==============================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ==============================
# 8. Training Loop
# ==============================
train_acc_hist, val_acc_hist = [], []

for epoch in range(10):  # you can increase to 20–30
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()
        total += labels.size(0)

    train_acc = correct / total
    train_acc_hist.append(train_acc)

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)

    val_acc = correct / total
    val_acc_hist.append(val_acc)

    print(f"📍 Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

# ==============================
# 9. Save Model
# ==============================
torch.save(model.state_dict(), "resnet50_food_classifier.pth")
print("💾 Model saved as resnet50_food_classifier.pth")

# ==============================
# 10. Plot Accuracy Curves
# ==============================
plt.plot(train_acc_hist, label="Train Acc")
plt.plot(val_acc_hist, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("ResNet50 Training vs Validation Accuracy")
plt.show()

📍 Epoch 1: Train Acc=0.4662, Val Acc=0.7072
📍 Epoch 2: Train Acc=0.7058, Val Acc=0.7350
📍 Epoch 3: Train Acc=0.7329, Val Acc=0.7482


In [1]:
import torch
import torch.nn as nn
from torchvision import models

# Food classes
classes = ['Aloo_Bhujia', 'Aloo_Gobi', 'Aloo_Matar', 'Baingan_Bharta', 'Bhindi_Masala',
           'Bhuna_Gosht', 'Chicken_Karahi', 'Chicken_Shorba', 'French_Fries', 'Gajar_ka_Halwa',
           'Gulab_Jamun', 'Haleem', 'Jalebi', 'Kaddu_Sabzi', 'Kheer', 'Lauki_Chana_Dal',
           'Mix_Sabzi', 'Naan', 'Paratha', 'Plain_White_Rice', 'Rabri', 'Ras_Malai',
           'Roll_Paratha', 'Roti', 'Samosa', 'Sheer_Khurma', 'Zeera_Rice']

# Rebuild model architecture
model = models.resnet50(weights=None)  # start with blank ResNet50
model.fc = nn.Linear(model.fc.in_features, len(classes))  # adjust final layer

# Load trained weights
model.load_state_dict(torch.load("resnet50_food_classifier.pth", map_location="cpu"))

# Set to evaluation mode
model.eval()
print("✅ Model loaded and ready for inference!")


FileNotFoundError: [Errno 2] No such file or directory: 'resnet50_food_classifier.pth'

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Load base model
model = models.resnet50(weights=None)  # or weights="IMAGENET1K_V1" if you want pretrained
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)  # binary classification

# Load only compatible layers
state_dict = torch.load("resnet50_food_classifier.pth", map_location="cpu")

# Remove last fc layer weights (since mismatch)
state_dict = {k: v for k, v in state_dict.items() if "fc" not in k}

model.load_state_dict(state_dict, strict=False)  # allow missing fc
model.eval()


In [ ]:
model = models.resnet50(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 27)

# Load full checkpoint
model.load_state_dict(torch.load("resnet50_food_classifier.pth", map_location="cpu"))
model.eval()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Your food classes
food_classes = [
    'Aloo_Bhujia', 'Aloo_Gobi', 'Aloo_Matar', 'Baingan_Bharta',
    'Bhindi_Masala', 'Bhuna_Gosht', 'Chicken_Karahi', 'Chicken_Shorba',
    'French_Fries', 'Gajar_ka_Halwa', 'Gulab_Jamun', 'Haleem',
    'Jalebi', 'Kaddu_Sabzi', 'Kheer', 'Lauki_Chana_Dal',
    'Mix_Sabzi', 'Naan', 'Paratha', 'Plain_White_Rice',
    'Rabri', 'Ras_Malai', 'Roll_Paratha', 'Roti',
    'Samosa', 'Sheer_Khurma', 'Zeera_Rice'
]

def predict_image(image_path):
    # Load and preprocess
    image = Image.open(image_path).convert('RGB')
    img_tensor = transform(image).unsqueeze(0)

    # Predict
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)

    pred_idx = predicted.item()
    predicted_class = food_classes[pred_idx]

    # Show image with prediction
    plt.figure(figsize=(5,5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Prediction: {pred_idx} → {predicted_class}", fontsize=14, color="black")
    plt.show()

    print(f"Predicted class index: {pred_idx}")
    print(f"Predicted dish name : {predicted_class}")

# Example
predict_image("/content/drive/MyDrive/TEST_IMAGE_10.jpeg")
